# W03: State Space Search (II)

Today we'll be focusing on state-space search and some simple search algorithms.

## Uninformed Search (from last lecture)

In the code below, I've provided you with some starter code that creates a "graph", represented as an adjacency matrix: each element in the matrix is a bool representing whether you can traverse directly from one node to another. The plot below shows that the adjacency matrix is a *band matrix*, so that only the diagonal band has non-zero elements. This means that there will not be a direct connection between the start and the goal: search is therefore needed. The adjacency matrix is symmetric, meaning that you can travel in both directions between all nodes.

**TASK** Finish implementing the `uninformed_search_for_goal` function by resolving the `NotImplementeError`s

**TASK** Add a "visited list" and see how the number of iterations goes down.

I have also provided you with a function `search_for_goal_scipy` that uses the Scipy package to produce a solution. Compare the two functions.

In [ ]:
# Starter Code: Graph definition and plotting

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import random


def get_random_adjacency(band_size, num_nodes, edge_rate=1.0, seed=695):
    random.seed(seed)
    mat = np.zeros((num_nodes, num_nodes))
    for ii in range(num_nodes):
        for jj in range(ii + 1, min(ii+band_size, num_nodes)):
            val = 1.0 * (random.random() < edge_rate)
            mat[ii, jj] = val
            mat[jj, ii] = val
    return mat


def plot_path_on_mat(mat, path=None, ax=None, cmap='viridis', title=None):
    """Show the matrix, with the path drawn as a 'bouncing ball' above it.

    The strip above the matrix is the list of nodes (same x axis as the
    matrix's columns): the ball starts at the start node and hops to each
    node of the path in turn, forward hops arcing above the line and any
    backtracking arcing below it.  Arcs run dark -> bright in `plasma` as
    the search progresses, so you can read off the order of traversal.

    On the matrix itself, an open circle marks each element actually
    traversed -- the *edge* mat[a, b], not the (empty) diagonal.
    """
    if ax is None:
        _, ax = plt.subplots(dpi=150)
    # Washed out on purpose: the matrix is the backdrop here, and every
    # traversed element is a *non-zero* one, so it needs to stay light
    # enough for a coloured ring to read against it.
    mat = mat.copy()
    mat[mat <= 0] = np.nan
    ax.imshow(mat, interpolation='none', cmap=cmap)
    ax.set_xticks([])
    ax.set_yticks([])

    if path is None:
        return

    # A strip above the matrix, sharing its x axis, to bounce the ball in.
    bax = make_axes_locatable(ax).append_axes('top', size='25%', pad=0.05,
                                              sharex=ax)
    bax.set_yticks([])
    bax.set_xticks([])
    for side in bax.spines.values():
        side.set_visible(False)
    if title is not None:
        bax.set_title(title, fontsize=9)

    if not path:
        return ax

    hops = list(zip(path[:-1], path[1:]))
    colors = plt.get_cmap('plasma')(np.linspace(0.0, 0.85, max(len(hops), 1)))
    halo = [path_effects.withStroke(linewidth=3, foreground='white')]

    # The baseline the ball bounces along: one tick per node of the graph.
    bax.axhline(0, color='0.8', linewidth=0.8, zorder=1)
    bax.plot(path, np.zeros(len(path)), '.', color='0.4', markersize=3, zorder=2)

    u = np.linspace(0, 1, 100)
    for (a, b), color in zip(hops, colors):
        height = (b - a)
        bax.plot(a + (b - a) * u, height * np.sin(np.pi * u),
                 color=color, linewidth=1.5, zorder=3, path_effects=halo)
        # ... and the element of the matrix that hop actually used.
        # ... and the element of the matrix that hop actually used.
        ax.plot(b, a, 'o', markerfacecolor='none', markeredgecolor=color,
                markersize=7, markeredgewidth=1.5, zorder=3, path_effects=halo)

    # Headroom below the baseline for the start/goal labels (and for any
    # backward hops, which arc downwards
    span = max(abs(b - a) for a, b in hops)
    bax.set_ylim(-1.15 * span if any(b < a for a, b in hops) else -0.22 * span,
                 1.15 * span)
    for name, node, va in [('start', path[0], 'top'), ('goal', path[-1], 'top')]:
        bax.annotate(name, (node, 0), color='0.3', fontsize=7, ha='center',
                     va=va, xytext=(0, -4), textcoords='offset points')
    return ax

    
def get_path_cost(mat, path):
    """Total cost of a path: the sum of the weights of the edges it uses.

    Zero means 'no edge' here, exactly as in `get_actions`, so a path that
    hops between unconnected nodes is a bug, not a free ride."""
    if not path:
        return np.inf
    cost = 0.0
    for a, b in zip(path[:-1], path[1:]):
        if mat[a, b] <= 0:
            raise ValueError(f"No edge between nodes {a} and {b}.")
        cost += mat[a, b]
    return float(cost)


mat = get_random_adjacency(band_size=10, num_nodes=40, edge_rate=0.3)
plot_path_on_mat(mat)

In [ ]:
# SortedList example (much faster than alternatives)

from sortedcontainers import SortedList

# Sort by second element
sl_example = SortedList(key=lambda e: e[1])
sl_example.add([1, 2])
sl_example.add([2, 3])
sl_example.add([3, 0])
sl_example.add([0, 4])
print(sl_example)

# Sort by first element
sl_example = SortedList(key=lambda e: e[0])
sl_example.add([1, 2])
sl_example.add([2, 3])
sl_example.add([3, 0])
sl_example.add([0, 4])
print(sl_example)

In [ ]:
## Starter Code: state space search

def get_actions(mat, state):
    """Get the available 'moves' from the current state.
    
    return: list of indices connected to 'state'."""
    return np.where(mat[state] > 0)[0].tolist()

print(f"Actions from node 0: {get_actions(mat,0)=}")

def uninformed_search_for_goal(start, goal, mat, max_iterations=100000):
    # Q is a list of lists (each sub-list is a path from the start)
    Q = SortedList([[start,]], key=lambda e: len(e))
    visited = set([start])
    
    for ii in range(max_iterations):
        N = None
        # Get (and remove) the (first|last)? element of Q.
        #       Which is for Depth/Breadth first?
        #   N = Q.pop(0)
        #   N = Q.pop(-1)
        if N is None:
            raise NotImplementedError("Pop an element off Q to get 'N'.")

        # If we have reached the goal, return
        if N[-1] == goal:
            print(f"  Num_iterations: {ii}")
            print(f"  Length of Q: {len(Q)}")
            return N
        
        # Expand Children: loop through all children and expand the path.
        for action in get_actions(mat, N[-1]):
            Q.add(N + [action])

    # We never found the goal.
    return None

# Scipy Solver Solution
from scipy.sparse.csgraph import shortest_path

def search_for_goal_scipy(start, goal, mat):
    dist_matrix, predecessors = shortest_path(csgraph=mat, method='D', directed=False, indices=start, return_predecessors=True)
    path = [goal]
    while True:
        if path[-1] == start:
            return path[::-1]
        if path[-1] < 0:
            return None
        path.append(predecessors[path[-1]])

In [ ]:
# Look into some solutions (feel free to change the parameters)

band_size = 10
num_nodes = 40
mat = get_random_adjacency(band_size, num_nodes, edge_rate=0.3)

print("Our solution:")
path = uninformed_search_for_goal(0, num_nodes - 1, mat)
path_scipy = search_for_goal_scipy(0, num_nodes - 1, mat)
print(f"Solution [len={len(path)}]: {path}")
print(f"Solution (scipy) [len={len(path_scipy)}]: {[int(e) for e in path_scipy]}")

fig, axs = plt.subplots(1, 2, dpi=150, figsize=(8, 4))
plot_path_on_mat(mat, path, ax=axs[0], title=f'ours (cost={get_path_cost(mat, path)})')
plot_path_on_mat(mat, path_scipy, ax=axs[1], title=f'scipy (cost={get_path_cost(mat, path_scipy)})')
None

## Informed Search (new!)

Next we'll look at *informed search* and add weights to the edges in our graph.

**TASK** Finish implementing the `informed_search_for_goal` function by resolving the `NotImplementeError`s

**TASK** Add an "expanded list" and see how the number of iterations goes down.

In [ ]:
# Starter Code: Graph definition and plotting

import matplotlib.pyplot as plt
import numpy as np
import random


def get_random_weight_mat(band_size, num_nodes, edge_rate=1.0, seed=695):
    random.seed(seed)
    mat = np.zeros((num_nodes, num_nodes))
    for ii in range(num_nodes):
        for jj in range(ii, min(ii+band_size, num_nodes)):
            val = abs(ii - jj) + random.random()
            mat[ii, jj] = val
            mat[jj, ii] = val
    return mat


def plot_weight_mat(mat):
    plt.figure(dpi=150)
    plt.xticks([], [])
    plt.yticks([], [])
    plt.imshow(mat, interpolation='none')

mat = get_random_weight_mat(band_size=10, num_nodes=40, edge_rate=0.3)
plot_path_on_mat(mat)

In [ ]:
def informed_search_for_goal(start, goal, mat, max_iterations=1000000):
    # Q is a list of lists (each sub-list is a path from the start)
    # Hint: elements of Q should be two-element tuples:
    #   1) the cost so far
    #   2) the path so far
    # raise NotImplementedError("Queue 'Q' not defined.")
    Q = SortedList([(0, [start])], key=lambda e: e[0])
    
    visited = set([start])
    
    for ii in range(max_iterations):
        # Get (and remove) the (first|last)? element of Q.
        #       Which is for Depth/Breadth first?
        #   N = Q.pop(0)
        #   N = Q.pop(-1)
        cost, N = Q.pop(0)

        # If we have reached the goal, return
        if N[-1] == goal:
            print(f"  Num_iterations: {ii}")
            print(f"  Length of Q: {len(Q)}")
            return N

        # TODO: what about the "visited" list? expanded list maybe?

        # Expand Children: loop through all children and expand the path.
        for next_state in get_actions(mat, N[-1]):
            action_cost = mat[N[-1], next_state]
            raise NotImplementedError("Define the new cost and add to queue.")

    # We never found the goal.print(f"  Num_iterations: {ii}")
    print(f"  Num_iterations: {ii}")
    print(f"  Length of Q: {len(Q)}")
    return None


In [ ]:
# Look into some solutions (feel free to change the parameters)

band_size = 3
num_nodes = 8
mat = get_random_weight_mat(band_size, num_nodes)
plot_path_on_mat(mat)

print("Our solution:")
path = informed_search_for_goal(0, num_nodes - 1, mat)
path_scipy = search_for_goal_scipy(0, num_nodes - 1, mat)
print(f"Solution [len={len(path)}]: {path}")
print(f"Solution (scipy) [len={len(path_scipy)}]: {[int(e) for e in path_scipy]}")

fig, axs = plt.subplots(1, 2, dpi=150, figsize=(8, 4))
plot_path_on_mat(mat, path, ax=axs[0], title=f'ours (cost={get_path_cost(mat, path):0.3f})')
plot_path_on_mat(mat, path_scipy, ax=axs[1], title=f'scipy (cost={get_path_cost(mat, path_scipy):0.3f})')
None

In [ ]:
#####################################
## Solution below; no looking yet! ##
#####################################

In [ ]:
def informed_search_for_goal(start, goal, mat, max_iterations=1000000):
    # Q is a list of lists (each sub-list is a path from the start)
    # Hint: elements of Q should be two-element tuples:
    #   1) the cost so far
    #   2) the path so far
    # raise NotImplementedError("Queue 'Q' not defined.")
    Q = SortedList([(0, [start])], key=lambda e: e[0])
    
    visited = set([start])
    
    for ii in range(max_iterations):
        # Get (and remove) the (first|last)? element of Q.
        #       Which is for Depth/Breadth first?
        #   N = Q.pop(0)
        #   N = Q.pop(-1)
        cost, N = Q.pop(0)

        # If we have reached the goal, return
        if N[-1] == goal:
            print(f"  Num_iterations: {ii}")
            print(f"  Length of Q: {len(Q)}")
            return N
        
        # Expand Children: loop through all children and expand the path.
        for next_state in get_actions(mat, N[-1]):
            new_cost = cost + mat[N[-1], next_state]
            Q.add((new_cost, N + [next_state]))

    # We never found the goal.
    return None

# Look into some solutions (feel free to change the parameters)
band_size = 3
num_nodes = 10
mat = get_random_weight_mat(band_size, num_nodes)

print("Our solution:")
path = informed_search_for_goal(0, num_nodes - 1, mat)
path_scipy = search_for_goal_scipy(0, num_nodes - 1, mat)
print(f"Solution [len={len(path)}]: {path}")
print(f"Solution (scipy) [len={len(path_scipy)}]: {[int(e) for e in path_scipy]}")

fig, axs = plt.subplots(1, 2, dpi=150, figsize=(8, 4))
plot_path_on_mat(mat, path, ax=axs[0], title=f'ours (cost={get_path_cost(mat, path):0.3f})')
plot_path_on_mat(mat, path_scipy, ax=axs[1], title=f'scipy (cost={get_path_cost(mat, path_scipy):0.3f})')
None